# ⚔️ DSWA: Nossa Primeira Competição Kaggle!

Bem-vindos ao nosso primeiro desafio prático! Hoje vamos participar da competição **Predict Customer Churn** no Kaggle.

**O Problema:** Uma empresa de telecomunicações quer saber quais clientes vão cancelar a assinatura (o que chamamos de *Churn*). Descobrir isso antes que o cliente vá embora permite que a empresa ofereça descontos ou vantagens para mantê-lo.

**Nosso Objetivo:** Ensinar uma Inteligência Artificial a olhar para o histórico dos clientes e prever a probabilidade de eles cancelarem o serviço!

Vamos fazer isso em 6 passos simples.

## Passo 1: Preparando a Caixa de Ferramentas 🧰

Antes de construir uma casa, precisamos de tijolos e cimento. No Python, usamos "bibliotecas", que são pacotes de códigos já prontos que nos ajudam a trabalhar com dados, criar gráficos e treinar a IA.

In [ ]:
# Importando nossas bibliotecas e ferramentas:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

## Passo 2: Conhecendo Nossos Dados 🔍
Vamos carregar o arquivo train.csv (os dados dos clientes que sabemos se cancelaram ou não) e dar uma olhada neles.

In [ ]:
PATH = './dataset/'
treino = pd.read_csv(PATH + 'train.csv')
teste = pd.read_csv(PATH + 'test.csv')

display(treino.head().style.hide(axis="index"))

In [ ]:
# Estatísticas descritivas do DataFrame
treino.describe(include='all')

In [ ]:
# Número de itens nulos por coluna
print(treino.isna().sum())

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=treino, x='Churn')
plt.title('Distribuição de Cancelamentos (Churn)')
plt.show()

plt.figure(figsize=(8, 5))
sns.histplot(data=treino, x='tenure', hue='Churn', multiple='stack', bins=30)
plt.title('Tempo de Contrato (meses) vs Cancelamento')
plt.show()

colunas_numericas = treino.select_dtypes(include=['number']).columns
correlacao = treino[colunas_numericas].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(correlacao, annot=True, fmt=".2f", cmap='coolwarm', square=True, linewidths=0.5)
plt.title('Mapa de Calor: Quais informações estão mais ligadas?')
plt.show()

## Adicionando nova coluna

In [ ]:
treino['monthlyExpense'] = treino['TotalCharges'] / treino['tenure']
teste['monthlyExpense'] = teste['TotalCharges'] / teste['tenure']

## Passo 3: Traduzindo para a Máquina (Pré-processamento) 🤖
O computador é muito rápido, mas ele é "bobo": ele só entende números. Ele não sabe o que é "Fibra Ótica" ou "Feminino". Precisamos transformar todo o texto em números.

In [ ]:
# Exemplo pequeno para demonstrar
exemplo_original = treino[['gender', 'InternetService']].head(5)

# Aplicando o One-Hot Encoding no exemplo
exemplo_encoded = pd.get_dummies(exemplo_original)

print("🠖 ANTES (Dados Categóricos)")
display(exemplo_original)

print("\n🠖 DEPOIS (One-Hot Encoding)")
display(exemplo_encoded)

In [ ]:
X = treino.drop(columns=['id', 'Churn'])
y = treino['Churn']

X_teste_comp = teste.drop(columns=['id'])
X = pd.get_dummies(X, drop_first=True)
X_teste_comp = pd.get_dummies(X_teste_comp, drop_first=True)

X, X_teste_comp = X.align(X_teste_comp, join='inner', axis=1)

print("Dados traduzidos para números com sucesso!")
display(X.head())

## Passo 4: O Treinamento (Machine Learning) 🧠
Antes de colocar nosso modelo para fazer a prova do Kaggle, precisamos testá-lo em casa.
Para isso, vamos separar nossos dados de treino: 80% para a IA estudar, e 20% para a gente fazer um simulado.

Por que separamos dados para o simulado? Porque se a IA estudar todas as questões, ela pode simplesmente 'decorar' as respostas em vez de aprender a lógica. O simulado com questões que ela nunca viu garante que ela realmente aprendeu a matéria!

Neste notebook, vamos usar o **Random Forest** (Floresta Aleatória). Imagine que, em vez de perguntar para um único especialista se o cliente vai cancelar, nós perguntamos para uma equipe de 100 especialistas diferentes e tiramos uma média da opinião deles. É isso que esse algoritmo faz!

In [ ]:
X_treino, X_valid, y_treino, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

le = LabelEncoder()
y_treino = le.fit_transform(y_treino)

# Trocando para usar XGBoost ao invés do Random Forest
modelo = XGBClassifier(
    n_estimators=100, 
    random_state=42
)

print("Treinando a IA...")
modelo.fit(X_treino, y_treino)
print("Treinamento concluído!")

## Passo 5: Avaliação do Modelo 📊
O Kaggle vai nos avaliar usando uma métrica chamada **ROC AUC**.

Pense assim: a IA dá uma nota de 0 a 100 para o risco de cada cliente cancelar. O ROC AUC avalia se a IA deu as notas mais altas realmente para as pessoas que cancelaram.
- **Nota 0.50:** A IA está apenas "chutando" (jogando uma moeda).
- **Nota 1.00:** A IA acertou perfeitamente.

In [ ]:
previsoes_simulado = modelo.predict_proba(X_valid)[:, 1]

nota_auc = roc_auc_score(y_valid, previsoes_simulado)
print("="*40)
print(f"🎉 NOSSA NOTA NO SIMULADO: {nota_auc:.4f} 🎉")
print("="*40)

Lembra dos 100 especialistas do Random Forest? Aqui está o relatório do que eles mais olharam na ficha do cliente para tomar uma decisão. Geralmente, o tempo de contrato (tenure) e o valor da mensalidade e anuidade são os fatores decisivos!

In [ ]:
# O que a IA achou mais importante?
importancias = pd.Series(modelo.feature_importances_, index=X.columns)
importancias = importancias.sort_values(ascending=False).head(10) # Pegando as 10 mais importantes

plt.figure(figsize=(10, 6))
sns.barplot(x=importancias.values, y=importancias.index, hue=importancias.index, palette='viridis', legend=False)
plt.title('O que a IA levou mais em conta para prever o Churn?')
plt.xlabel('Grau de Importância')
plt.ylabel('')
plt.show()

## Passo 6: Gerando o Arquivo para o Kaggle! 🚀

Agora que vimos que o modelo aprendeu alguma coisa no simulado, vamos usá-lo para prever o arquivo ```test.csv``` oficial do Kaggle.

In [ ]:
previsoes_finais = modelo.predict_proba(X_teste_comp)[:, 1]

submissao = pd.DataFrame({
    'id': teste['id'],
    'Churn': previsoes_finais
})

submissao.to_csv('submission.csv', index=False)
print("Arquivo salvo! Agora é só enviar no site do Kaggle!")
# Para submeter, vá na aba 'Submit to competition' na direita e clique no botão 'Submit'

## 💡 Desafio para os Membros: Como subir no Ranking?

Esse notebook é apenas o nosso ponto de partida (nosso "Baseline"). A nota que tiramos não é a máxima possível. O verdadeiro trabalho de um Cientista de Dados começa agora!

Aqui estão algumas estratégias legais (e simples) que vocês podem tentar sozinhos para melhorar o modelo e conseguir uma pontuação maior:

1. **Feature Engineering (Engenharia de Variáveis):** Às vezes, criar uma nova coluna ajuda o modelo. Por exemplo, criar uma coluna "Cobrança Mensal por Mês de Contrato" dividindo as taxas pelo tempo de uso (```tenure```).

2. **Brincar com os Hiperparâmetros:** No ```RandomForestClassifier(n_estimators=100)```, o que acontece se você mudar para ```n_estimators=500```? Ou se limitar a profundidade da árvore de decisão com ```max_depth=5```? O modelo melhora ou piora?

3. **Mudar a IA:** Nós usamos Random Forest. Que tal importar e testar o ```LogisticRegression``` ou o ```XGBoost```? Algoritmos diferentes aprendem de jeitos diferentes.

4. **Balanceando o modelo:** Como temos muito mais dados de quem não cancelou, sua IA pode ter dificuldade em aprender os padrões de quem sai. Que tal testar o parâmetro ```class_weight='balanced'``` para forçar o modelo a dar a mesma importância aos dois grupos?

*Escolha uma dessas ideias, altere o código acima e veja se sua nota sobe! Boa sorte!*